# 01 — Data Collection: Dataset ASEAN Multi-Negara

Notebook ini mengumpulkan dan menyiapkan data ekonomi dari **10 negara ASEAN** yang diunduh dari World Bank.

## Negara yang Dicakup (10 Negara)
Brunei, Indonesia, Kamboja, Laos, Malaysia, Myanmar, Filipina, Singapura, Thailand, Vietnam

## Rentang Tahun
1961 – 2024

## Indikator yang Digunakan
| Kolom | Indikator World Bank |
|-------|----------------------|
| `GDP_Growth` | GDP growth (annual %) — NY.GDP.MKTP.KD.ZG |
| `GDP_Per_Capita` | GDP per capita (current US$) — NY.GDP.PCAP.CD |
| `Population_Growth` | Population growth (annual %) — SP.POP.GROW |
| `Exports_pct` | Exports of goods and services (% of GDP) — NE.EXP.GNFS.ZS |
| `Imports_pct` | Imports of goods and services (% of GDP) — NE.IMP.GNFS.ZS |
| `Life_Expectancy` | Life expectancy at birth, total (years) — SP.DYN.LE00.IN |

## Alur Pipeline
`data/raw/dataset_asean.csv` (World Bank) → rename kolom → konversi tipe → rename negara → simpan kembali

In [26]:
# =====================================================================
# SEL 1: Import library
# =====================================================================

import pandas as pd
import numpy as np
import os

print('✅ Library siap.')

✅ Library siap.


In [27]:
# =====================================================================
# SEL 2: Load file CSV mentah dari World Bank
# =====================================================================

import sys, os

# Deteksi otomatis root folder proyek
current_path = os.getcwd()
while current_path:
    if os.path.exists(os.path.join(current_path, 'scripts')):
        break
    parent = os.path.dirname(current_path)
    if parent == current_path:
        current_path = None
        break
    current_path = parent

if not current_path:
    raise RuntimeError('❌ Root folder proyek tidak ditemukan.')

RAW_PATH = os.path.join(current_path, 'data', 'raw', 'dataset_asean.csv')

if not os.path.exists(RAW_PATH):
    raise FileNotFoundError(
        '❌ File tidak ditemukan: ' + RAW_PATH + '\n'
        'Pastikan file CSV sudah ada di folder data/raw/'
    )

df_raw = pd.read_csv(RAW_PATH, na_values='..')

print('✅ File berhasil dibaca.')
print(f'   Path       : {RAW_PATH}')
print(f'   Shape awal : {df_raw.shape[0]} baris x {df_raw.shape[1]} kolom')
print(f'   Kolom      : {df_raw.columns.tolist()}')

✅ File berhasil dibaca.
   Path       : c:\Users\fsyah\Kelompok1-ML\data\raw\dataset_asean.csv
   Shape awal : 655 baris x 10 kolom
   Kolom      : ['Time', 'Time Code', 'Country Name', 'Country Code', 'GDP growth (annual %) [NY.GDP.MKTP.KD.ZG]', 'GDP per capita (current US$) [NY.GDP.PCAP.CD]', 'Population growth (annual %) [SP.POP.GROW]', 'Exports of goods and services (% of GDP) [NE.EXP.GNFS.ZS]', 'Imports of goods and services (% of GDP) [NE.IMP.GNFS.ZS]', 'Life expectancy at birth, total (years) [SP.DYN.LE00.IN]']


In [28]:
# =====================================================================
# SEL 3: Rename kolom ke nama standar proyek
# =====================================================================

rename_dict = {
    'Country Name': 'Country',
    'Time'        : 'Year',
    'GDP growth (annual %) [NY.GDP.MKTP.KD.ZG]'                        : 'GDP_Growth',
    'GDP per capita (current US$) [NY.GDP.PCAP.CD]'                    : 'GDP_Per_Capita',
    'Population growth (annual %) [SP.POP.GROW]'                       : 'Population_Growth',
    'Exports of goods and services (% of GDP) [NE.EXP.GNFS.ZS]'       : 'Exports_pct',
    'Imports of goods and services (% of GDP) [NE.IMP.GNFS.ZS]'       : 'Imports_pct',
    'Life expectancy at birth, total (years) [SP.DYN.LE00.IN]'         : 'Life_Expectancy'
}

df_raw.rename(columns=rename_dict, inplace=True)

print('✅ Rename kolom selesai.')
print(f'   Kolom baru: {df_raw.columns.tolist()}')

✅ Rename kolom selesai.
   Kolom baru: ['Year', 'Time Code', 'Country', 'Country Code', 'GDP_Growth', 'GDP_Per_Capita', 'Population_Growth', 'Exports_pct', 'Imports_pct', 'Life_Expectancy']


In [29]:
# =====================================================================
# SEL 4: Konversi tipe data kolom numerik
# =====================================================================

numeric_cols = [
    'GDP_Growth', 'GDP_Per_Capita', 'Population_Growth',
    'Exports_pct', 'Imports_pct', 'Life_Expectancy'
]

for col in numeric_cols:
    if col in df_raw.columns:
        df_raw[col] = pd.to_numeric(df_raw[col], errors='coerce')

# Konversi Year ke integer
df_raw['Year'] = pd.to_numeric(df_raw['Year'], errors='coerce').astype('Int64')

print('✅ Konversi tipe data selesai.')
print(f'   Tipe Year: {df_raw["Year"].dtype}')

✅ Konversi tipe data selesai.
   Tipe Year: Int64


In [30]:
# =====================================================================
# SEL 5: Standarisasi nama negara
# =====================================================================

df_raw['Country'] = df_raw['Country'].replace({
    'Brunei Darussalam': 'Brunei',
    'Lao PDR'          : 'Laos',
    'Viet Nam'         : 'Vietnam',
    'Khmer Republic'   : 'Cambodia',
    'Cambodia'         : 'Cambodia',
    'Philippines'      : 'Philippines',
    'Singapore'        : 'Singapore',
    'Myanmar'          : 'Myanmar',
    'Indonesia'        : 'Indonesia',
    'Malaysia'         : 'Malaysia',
    'Thailand'         : 'Thailand'
})

# Hapus baris yang kolom Country-nya kosong/NaN (biasanya baris footer dari World Bank)
df_raw = df_raw.dropna(subset=['Country'])

print('✅ Standarisasi nama negara selesai.')
print(f'   Negara tersedia: {sorted(df_raw["Country"].unique().tolist())}')

✅ Standarisasi nama negara selesai.
   Negara tersedia: ['Brunei', 'Cambodia', 'Indonesia', 'Laos', 'Malaysia', 'Myanmar', 'Philippines', 'Singapore', 'Thailand', 'Vietnam']


In [31]:
# =====================================================================
# SEL 6: Pilih kolom yang diperlukan & simpan
# =====================================================================

cols_needed = ['Country', 'Year'] + numeric_cols
df_asean_clean = df_raw[cols_needed].copy()

df_asean_clean.to_csv(RAW_PATH, index=False)

print('✅ File berhasil disimpan ke: ' + RAW_PATH)

✅ File berhasil disimpan ke: c:\Users\fsyah\Kelompok1-ML\data\raw\dataset_asean.csv


In [32]:
# =====================================================================
# SEL 7: Validasi hasil — negara, tahun, shape
# =====================================================================

print('=' * 55)
print('VALIDASI DATASET ASEAN')
print('=' * 55)
print(f'Shape            : {df_asean_clean.shape[0]} baris x {df_asean_clean.shape[1]} kolom')
print(f'Rentang tahun    : {df_asean_clean["Year"].min()} — {df_asean_clean["Year"].max()}')
print(f'Negara tersedia  : {sorted(df_asean_clean["Country"].unique().tolist())}')
print()

# Cek jumlah negara (target: 10 negara ASEAN)
n_countries = df_asean_clean['Country'].nunique()
if n_countries >= 10:
    print(f'✅ Jumlah negara memenuhi syarat: {n_countries} negara')
else:
    print(f'⚠️  Jumlah negara kurang: {n_countries} (target 10 negara ASEAN)')

# Cek rentang tahun (target: 1961–2024)
if df_asean_clean['Year'].min() <= 1961 and df_asean_clean['Year'].max() >= 2024:
    print('✅ Rentang tahun memenuhi syarat (1961–2024)')
else:
    print(f'⚠️  Rentang tahun: {df_asean_clean["Year"].min()}–{df_asean_clean["Year"].max()} (target 1961–2024)')

# Cek kolom Country ada
if 'Country' in df_asean_clean.columns:
    print('✅ Kolom Country tersedia')
else:
    print('❌ Kolom Country tidak ditemukan!')

VALIDASI DATASET ASEAN
Shape            : 650 baris x 8 kolom
Rentang tahun    : 1961 — 2025
Negara tersedia  : ['Brunei', 'Cambodia', 'Indonesia', 'Laos', 'Malaysia', 'Myanmar', 'Philippines', 'Singapore', 'Thailand', 'Vietnam']

✅ Jumlah negara memenuhi syarat: 10 negara
✅ Rentang tahun memenuhi syarat (1961–2024)
✅ Kolom Country tersedia


In [33]:
# =====================================================================
# SEL 8: Pratinjau & ringkasan missing values
# =====================================================================

print('5 baris pertama:')
display(df_asean_clean.head())

print('\nMissing values per kolom (wajar ada — akan diisi saat preprocessing):')
missing = df_asean_clean[numeric_cols].isnull().sum()
missing_pct = (missing / len(df_asean_clean) * 100).round(1)
summary = pd.DataFrame({'Jumlah NaN': missing, 'Persen (%)': missing_pct})
print(summary)

5 baris pertama:


,Country,Year,GDP_Growth,GDP_Per_Capita,Population_Growth,Exports_pct,Imports_pct,Life_Expectancy
0,Brunei,1961,NaN,NaN,4.612017,NaN,NaN,59.387
1,Indonesia,1961,5.740646,NaN,2.786731,11.061476,13.614125,47.385
2,Malaysia,1961,7.597994,232.943769,2.583738,48.471316,41.583648,58.015
3,Vietnam,1961,NaN,NaN,2.660493,NaN,NaN,58.461
4,Thailand,1961,5.362146,109.728794,2.930692,17.335476,17.110096,51.065



Missing values per kolom (wajar ada — akan diisi saat preprocessing):
                   Jumlah NaN  Persen (%)
GDP_Growth                 87        13.4
GDP_Per_Capita             81        12.5
Population_Growth          10         1.5
Exports_pct               200        30.8
Imports_pct               200        30.8
Life_Expectancy            10         1.5
